# Cálculo de Indicadores y KPIs — e-Visor · Ecocampus UPB Medellín

**Ref. especificación:** Indicadores_y_KPI_26_1_3  
**Autor:** Nicolás Meneses Marín  
**Fecha:** 2026-06-07  

---

> ⚠ **Parámetros bloqueantes (DEMO):** `N_USUARIOS_ACTIVOS` · `P_INSTALADA_KWP` · datos de inversores FV/baterías/sensor irradiancia · línea base KPI-04.  
> Los indicadores bloqueados se marcan con `⚠ PENDIENTE` y **no cortan la ejecución**.

> **Nota de unidades (crítica):** `activepower` está en **W** (no kW) y `activeenergyimport` es un **contador acumulado** en Wh. Las conversiones se aplican en cada sección.

In [1]:
import pandas as pd
import numpy as np
import openpyxl
import warnings, re
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Librerías cargadas.")

Librerías cargadas.


In [2]:
# Factor de emisión SIN colombiano (XM, 30-ene-2026)
FE_2025 = 0.000097018  # tCO2e/kWh — reemplaza el legacy 0.18 tCO2/MWh

# Umbrales técnicos (norma)
UMBRAL_DESBALANCE_OBJETIVO = 2.0   # % IEEE 1159:2019 / NEMA MG-1
UMBRAL_DESBALANCE_ALERTA   = 3.0   # %
UMBRAL_FP_OBJETIVO         = 0.90  # CREG Res. 108/1997
UMBRAL_FP_ALERTA           = 0.85

# Umbrales orientadores (bibliografía)
UMBRAL_LF_ORIENTADOR = 0.65   # Papadopoulos et al., 2016
UMBRAL_SS_ORIENTADOR = 15.0   # % GRI 302-1 / Ley 2169/2021
UMBRAL_PR_OBJETIVO   = 75.0   # % IEC 61724-1:2017 tropical
UMBRAL_PR_ALERTA     = 65.0   # %

# Áreas construidas por bloque (m²) — AREAS 2026.xlsx
AREAS_BLOQUE = {
    3:   4_778.68,
    4:  10_309.89,
    5:  10_008.87,
    7:   4_834.72,
    8:   3_836.47,
    9:   7_579.50,
   10:  11_469.06,
   12:   2_848.88,
   15:   7_780.01,
   17:   7_611.12,
   18:  35_916.80,
}

P_INSTALADA_KWP   = {}   # PENDIENTE: kWp Bloques 10 y 11
N_USUARIOS_ACTIVOS = None  # PENDIENTE: Rectoría / Registro académico

HORA_INICIO_OPERACION = 6   # 06:00 inclusive
HORA_FIN_OPERACION    = 21  # 21:59 inclusive
print("Parámetros configurados.")

Parámetros configurados.


In [3]:
def flag_pendiente(nombre, razon):
    print(f'\033[38;5;208m⚠ PENDIENTE — {nombre}:\n  {razon}\033[0m')
    return None

def extract_bloque(entity_id: str):
    m = re.search(r'_B(\d+)_', entity_id)
    return int(m.group(1)) if m else None

def estado_icon(value, objetivo, alerta, higher_is_better=True):
    if pd.isna(value): return '?'
    if higher_is_better:
        return 'OK' if value >= objetivo else ('AVISO' if value >= alerta else 'ALERTA')
    else:
        return 'OK' if value <= objetivo else ('AVISO' if value <= alerta else 'ALERTA')

print("Helpers definidos.")

Helpers definidos.


## 2. Carga y preparación del CSV

In [4]:
CSV_PATH = Path('etsmartmeter_clean.csv')
df_raw = pd.read_csv(CSV_PATH)
df_raw['time_index_colombia'] = pd.to_datetime(df_raw['time_index_colombia'])

# Zona horaria
if df_raw['time_index_colombia'].dt.tz is None:
    df_raw['time_index_colombia'] = df_raw['time_index_colombia'].dt.tz_localize(
        'America/Bogota', ambiguous='infer', nonexistent='shift_forward')
else:
    df_raw['time_index_colombia'] = df_raw['time_index_colombia'].dt.tz_convert('America/Bogota')

# Columnas auxiliares
df_raw['hora']   = df_raw['time_index_colombia'].dt.hour
df_raw['fecha']  = df_raw['time_index_colombia'].dt.date
df_raw['mes']    = df_raw['time_index_colombia'].dt.to_period('M')
df_raw['franja'] = np.where(
    (df_raw['hora'] >= HORA_INICIO_OPERACION) & (df_raw['hora'] <= HORA_FIN_OPERACION),
    'operacional', 'no_operacional')
df_raw['bloque'] = df_raw['entity_id'].apply(extract_bloque)

print(f'Shape: {df_raw.shape}')
print(f'Fechas: {df_raw["time_index_colombia"].min()} → {df_raw["time_index_colombia"].max()}')
print(f'Medidores ({df_raw["entity_id"].nunique()}):')
for eid in sorted(df_raw['entity_id'].unique()):
    b = df_raw.loc[df_raw['entity_id']==eid,'bloque'].iloc[0]
    print(f'  {eid}  →  bloque {b}')
print('\nNulos:')
print(df_raw[['activepower','activeenergyimport','v1','v2','v3',
              'totalpowerfactor','relativethdvoltage']].isnull().sum())

Shape: (42692, 14)
Fechas: 2026-01-30 19:00:00-05:00 → 2026-06-06 02:00:00-05:00
Medidores (16):
  SmartMeter_SM_B10_ARQ  →  bloque 10.0
  SmartMeter_SM_B12_DERE  →  bloque 12.0
  SmartMeter_SM_B15_BIBL  →  bloque 15.0
  SmartMeter_SM_B17_POLI  →  bloque 17.0
  SmartMeter_SM_B18_PARQ  →  bloque 18.0
  SmartMeter_SM_B3_RECT  →  bloque 3.0
  SmartMeter_SM_B4_PRIM  →  bloque 4.0
  SmartMeter_SM_B5_BACH  →  bloque 5.0
  SmartMeter_SM_B7_CTIC  →  bloque 7.0
  SmartMeter_SM_B7_TAC  →  bloque 7.0
  SmartMeter_SM_B8_AA  →  bloque 8.0
  SmartMeter_SM_B8_CPA  →  bloque 8.0
  SmartMeter_SM_B8_LABS  →  bloque 8.0
  SmartMeter_SM_B9_SFA1  →  bloque 9.0
  SmartMeter_SM_B9_SFA2  →  bloque 9.0
  SmartMeter_SM_ECOVILLA  →  bloque nan

Nulos:
activepower           0
activeenergyimport    0
v1                    0
v2                    0
v3                    0
totalpowerfactor      0
relativethdvoltage    0
dtype: int64


## 3. Preparación de datos derivados

- `activepower` (W) → `activepower_kw` (kW): dividir entre 1 000.
- `activeenergyimport` es contador acumulado en Wh → energía horaria con `diff()` por medidor.
- Energía diaria por medidor: suma de incrementos horarios del día.
- Potencia de bloque: suma de todos los medidores del mismo bloque por timestamp.

In [5]:
df = df_raw.sort_values(['entity_id', 'time_index_colombia']).copy()

# Energía horaria por medidor (Wh)
df['E_hora_wh'] = (
    df.groupby('entity_id', observed=True)['activeenergyimport']
    .diff().clip(lower=0)   # clip: resets o saltos → 0
)

# Potencia en kW
df['activepower_kw'] = df['activepower'] / 1000.0

# Energía diaria por medidor (kWh)
daily_meter = (
    df.groupby(['entity_id','bloque','fecha','mes'], observed=True)
    ['E_hora_wh'].sum().reset_index().rename(columns={'E_hora_wh':'E_dia_wh'})
)
daily_meter['E_dia_kwh'] = daily_meter['E_dia_wh'] / 1000.0

# Potencia por bloque (suma de medidores, por timestamp)
block_power = (
    df.groupby(['bloque','time_index_colombia','hora','fecha','mes','franja'], observed=True)
    ['activepower_kw'].sum().reset_index().rename(columns={'activepower_kw':'P_bloque_kw'})
)

# Energía diaria por bloque (suma de medidores)
daily_bloque = (
    daily_meter.groupby(['bloque','fecha','mes'], observed=True)
    ['E_dia_kwh'].sum().reset_index()
)

# Franja operacional / no operacional separadas
op     = block_power[block_power['franja'] == 'operacional']
non_op = block_power[block_power['franja'] == 'no_operacional']

print('daily_meter:', daily_meter.shape)
print('block_power:', block_power.shape)
print('daily_bloque:', daily_bloque.shape)

daily_meter: (1785, 6)
block_power: (29133, 7)
daily_bloque: (1301, 4)


## 4. INDICADORES

### IND-01 — LF (Load Factor)
**Fórmula:** `LF = mean(P) / max(P)` · diario por bloque + resumen mensual

In [6]:
ind01_daily = (
    block_power.groupby(['bloque','fecha','mes'], observed=True)
    .agg(P_mean=('P_bloque_kw','mean'), P_max=('P_bloque_kw','max')).reset_index()
)
ind01_daily['LF'] = ind01_daily['P_mean'] / ind01_daily['P_max']

ind01_monthly = (
    block_power.groupby(['bloque','mes'], observed=True)
    .agg(P_mean=('P_bloque_kw','mean'), P_max=('P_bloque_kw','max')).reset_index()
)
ind01_monthly['LF'] = ind01_monthly['P_mean'] / ind01_monthly['P_max']
ind01_monthly['estado'] = ind01_monthly['LF'].apply(
    lambda v: estado_icon(v, UMBRAL_LF_ORIENTADOR, UMBRAL_LF_ORIENTADOR))

print(ind01_monthly[['bloque','mes','P_mean','P_max','LF','estado']].to_string(index=False))

 bloque     mes  P_mean    P_max     LF estado
 3.0000 2026-01  2.4917   4.4463 0.5604 ALERTA
 3.0000 2026-02  4.3627  11.7887 0.3701 ALERTA
 3.0000 2026-03  4.7442  15.1171 0.3138 ALERTA
 3.0000 2026-04  5.0730  13.4860 0.3762 ALERTA
 3.0000 2026-05  5.1069  16.2394 0.3145 ALERTA
 3.0000 2026-06  6.0198  13.9731 0.4308 ALERTA
 4.0000 2026-01 15.5639  19.2148 0.8100     OK
 4.0000 2026-02 23.6559  59.2032 0.3996 ALERTA
 4.0000 2026-03 21.6001  61.0139 0.3540 ALERTA
 4.0000 2026-04 21.4545  54.2526 0.3955 ALERTA
 4.0000 2026-05 21.3376  63.4569 0.3363 ALERTA
 4.0000 2026-06 25.3486  53.5272 0.4736 ALERTA
 5.0000 2026-01 15.4621  21.1488 0.7311     OK
 5.0000 2026-02 26.6565  71.9180 0.3707 ALERTA
 5.0000 2026-03 25.2601  72.2246 0.3497 ALERTA
 5.0000 2026-04 29.8461  80.0197 0.3730 ALERTA
 5.0000 2026-05 29.0247  85.8299 0.3382 ALERTA
 5.0000 2026-06 34.7669  79.2486 0.4387 ALERTA
 7.0000 2026-01 42.3297  50.6799 0.8352     OK
 7.0000 2026-02 50.3756  82.9221 0.6075 ALERTA
 7.0000 2026-

### IND-02 — PAR (Peak-to-Average Ratio)
**Fórmula:** `PAR = max(P) / mean(P)` (inversa del LF)

In [7]:
ind02_daily = ind01_daily[['bloque','fecha','mes','P_mean','P_max']].copy()
ind02_daily['PAR'] = ind02_daily['P_max'] / ind02_daily['P_mean']

ind02_monthly = ind01_monthly[['bloque','mes','P_mean','P_max']].copy()
ind02_monthly['PAR'] = ind02_monthly['P_max'] / ind02_monthly['P_mean']
print(ind02_monthly[['bloque','mes','PAR']].to_string(index=False))

 bloque     mes    PAR
 3.0000 2026-01 1.7844
 3.0000 2026-02 2.7022
 3.0000 2026-03 3.1864
 3.0000 2026-04 2.6584
 3.0000 2026-05 3.1799
 3.0000 2026-06 2.3212
 4.0000 2026-01 1.2346
 4.0000 2026-02 2.5027
 4.0000 2026-03 2.8247
 4.0000 2026-04 2.5287
 4.0000 2026-05 2.9739
 4.0000 2026-06 2.1116
 5.0000 2026-01 1.3678
 5.0000 2026-02 2.6980
 5.0000 2026-03 2.8592
 5.0000 2026-04 2.6811
 5.0000 2026-05 2.9571
 5.0000 2026-06 2.2794
 7.0000 2026-01 1.1973
 7.0000 2026-02 1.6461
 7.0000 2026-03 1.5827
 7.0000 2026-04 1.7117
 7.0000 2026-05 1.7813
 7.0000 2026-06 1.5091
 8.0000 2026-01 1.3403
 8.0000 2026-02 2.7738
 8.0000 2026-03 2.7869
 8.0000 2026-04 2.6955
 8.0000 2026-05 2.7354
 8.0000 2026-06 2.5596
 9.0000 2026-01 1.4760
 9.0000 2026-02 2.4324
 9.0000 2026-03 2.3405
 9.0000 2026-04 2.3854
 9.0000 2026-05 2.9165
 9.0000 2026-06 1.5807
10.0000 2026-01 2.8357
10.0000 2026-02 2.8320
10.0000 2026-03 3.3997
10.0000 2026-04 3.0344
10.0000 2026-05 2.8778
10.0000 2026-06 2.0798
12.0000 202

### IND-03 — f₁ (Uniformidad operacional)
**Fórmula:** `f₁ = mean(P_op) / max(P_op)` · franja 06:00–21:59

In [8]:
ind03_daily = (
    op.groupby(['bloque','fecha','mes'], observed=True)
    .agg(P_op_mean=('P_bloque_kw','mean'), P_op_max=('P_bloque_kw','max')).reset_index()
)
ind03_daily['f1'] = ind03_daily['P_op_mean'] / ind03_daily['P_op_max']

ind03_monthly = (
    op.groupby(['bloque','mes'], observed=True)
    .agg(P_op_mean=('P_bloque_kw','mean'), P_op_max=('P_bloque_kw','max')).reset_index()
)
ind03_monthly['f1'] = ind03_monthly['P_op_mean'] / ind03_monthly['P_op_max']
print(ind03_monthly[['bloque','mes','f1']].to_string(index=False))

 bloque     mes     f1
 3.0000 2026-01 0.6529
 3.0000 2026-02 0.4880
 3.0000 2026-03 0.4122
 3.0000 2026-04 0.4958
 3.0000 2026-05 0.4149
 3.0000 2026-06 0.5858
 4.0000 2026-01 0.8582
 4.0000 2026-02 0.4846
 4.0000 2026-03 0.4253
 4.0000 2026-04 0.4789
 4.0000 2026-05 0.4034
 4.0000 2026-06 0.5996
 5.0000 2026-01 0.8041
 5.0000 2026-02 0.4743
 5.0000 2026-03 0.4456
 5.0000 2026-04 0.4781
 5.0000 2026-05 0.4300
 5.0000 2026-06 0.5765
 7.0000 2026-01 0.8662
 7.0000 2026-02 0.6665
 7.0000 2026-03 0.6909
 7.0000 2026-04 0.6491
 7.0000 2026-05 0.6293
 7.0000 2026-06 0.7530
 8.0000 2026-01 0.8491
 8.0000 2026-02 0.4354
 8.0000 2026-03 0.4315
 8.0000 2026-04 0.4470
 8.0000 2026-05 0.4341
 8.0000 2026-06 0.4835
 9.0000 2026-01 0.7555
 9.0000 2026-02 0.5095
 9.0000 2026-03 0.5357
 9.0000 2026-04 0.5239
 9.0000 2026-05 0.4304
 9.0000 2026-06 0.7416
10.0000 2026-01 0.4143
10.0000 2026-02 0.4298
10.0000 2026-03 0.3663
10.0000 2026-04 0.4027
10.0000 2026-05 0.4346
10.0000 2026-06 0.5784
12.0000 202

### IND-04 — f₂ (Coeficiente de variación de carga)
**Fórmula:** `f₂ = std(P_op) / mean(P_op)` · franja 06:00–21:59

In [9]:
ind04_daily = (
    op.groupby(['bloque','fecha','mes'], observed=True)
    .agg(P_op_std=('P_bloque_kw','std'), P_op_mean=('P_bloque_kw','mean')).reset_index()
)
ind04_daily['f2'] = ind04_daily['P_op_std'] / ind04_daily['P_op_mean']

ind04_monthly = (
    op.groupby(['bloque','mes'], observed=True)
    .agg(P_op_std=('P_bloque_kw','std'), P_op_mean=('P_bloque_kw','mean')).reset_index()
)
ind04_monthly['f2'] = ind04_monthly['P_op_std'] / ind04_monthly['P_op_mean']
print(ind04_monthly[['bloque','mes','f2']].to_string(index=False))

 bloque     mes     f2
 3.0000 2026-01 0.3497
 3.0000 2026-02 0.5667
 3.0000 2026-03 0.6082
 3.0000 2026-04 0.5540
 3.0000 2026-05 0.6498
 3.0000 2026-06 0.4114
 4.0000 2026-01 0.1063
 4.0000 2026-02 0.5320
 4.0000 2026-03 0.5775
 4.0000 2026-04 0.5248
 4.0000 2026-05 0.6407
 4.0000 2026-06 0.4368
 5.0000 2026-01 0.1597
 5.0000 2026-02 0.5475
 5.0000 2026-03 0.6013
 5.0000 2026-04 0.5403
 5.0000 2026-05 0.6545
 5.0000 2026-06 0.4171
 7.0000 2026-01 0.0860
 7.0000 2026-02 0.2030
 7.0000 2026-03 0.2177
 7.0000 2026-04 0.2314
 7.0000 2026-05 0.2428
 7.0000 2026-06 0.1920
 8.0000 2026-01 0.1216
 8.0000 2026-02 0.4302
 8.0000 2026-03 0.4789
 8.0000 2026-04 0.4414
 8.0000 2026-05 0.4443
 8.0000 2026-06 0.3657
 9.0000 2026-01 0.2259
 9.0000 2026-02 0.3765
 9.0000 2026-03 0.3960
 9.0000 2026-04 0.3731
 9.0000 2026-05 0.4167
 9.0000 2026-06 0.1357
10.0000 2026-01 0.4156
10.0000 2026-02 0.4621
10.0000 2026-03 0.5536
10.0000 2026-04 0.5229
10.0000 2026-05 0.4917
10.0000 2026-06 0.4845
12.0000 202

### IND-05 — f₃ (Min-to-mean)
**Fórmula:** `f₃ = min(P_op) / mean(P_op)` · franja 06:00–21:59

In [10]:
ind05_daily = (
    op.groupby(['bloque','fecha','mes'], observed=True)
    .agg(P_op_min=('P_bloque_kw','min'), P_op_mean=('P_bloque_kw','mean')).reset_index()
)
ind05_daily['f3'] = ind05_daily['P_op_min'] / ind05_daily['P_op_mean']

ind05_monthly = (
    op.groupby(['bloque','mes'], observed=True)
    .agg(P_op_min=('P_bloque_kw','min'), P_op_mean=('P_bloque_kw','mean')).reset_index()
)
ind05_monthly['f3'] = ind05_monthly['P_op_min'] / ind05_monthly['P_op_mean']
print(ind05_monthly[['bloque','mes','f3']].to_string(index=False))

 bloque     mes     f3
 3.0000 2026-01 0.4442
 3.0000 2026-02 0.1996
 3.0000 2026-03 0.1920
 3.0000 2026-04 0.2141
 3.0000 2026-05 0.1843
 3.0000 2026-06 0.2417
 4.0000 2026-01 0.7687
 4.0000 2026-02 0.1903
 4.0000 2026-03 0.2826
 4.0000 2026-04 0.2685
 4.0000 2026-05 0.1666
 4.0000 2026-06 0.3791
 5.0000 2026-01 0.7752
 5.0000 2026-02 0.2747
 5.0000 2026-03 0.3004
 5.0000 2026-04 0.2921
 5.0000 2026-05 0.2825
 5.0000 2026-06 0.3645
 7.0000 2026-01 0.8812
 7.0000 2026-02 0.6497
 7.0000 2026-03 0.5598
 7.0000 2026-04 0.1807
 7.0000 2026-05 0.6134
 7.0000 2026-06 0.6332
 8.0000 2026-01 0.7689
 8.0000 2026-02 0.3195
 8.0000 2026-03 0.2974
 8.0000 2026-04 0.3528
 8.0000 2026-05 0.3514
 8.0000 2026-06 0.5307
 9.0000 2026-01 0.6420
 9.0000 2026-02 0.2801
 9.0000 2026-03 0.3121
 9.0000 2026-04 0.2879
 9.0000 2026-05 0.2353
 9.0000 2026-06 0.6973
10.0000 2026-01 0.4970
10.0000 2026-02 0.1115
10.0000 2026-03 0.0298
10.0000 2026-04 0.0735
10.0000 2026-05 0.0998
10.0000 2026-06 0.3158
12.0000 202

### IND-06 — f₄ (Factor de carga no operacional)
**Fórmula:** `f₄ = mean(P_no_op) / mean(P_op)`

In [11]:
_nop_m = (
    non_op.groupby(['bloque','fecha','mes'], observed=True)
    ['P_bloque_kw'].mean().reset_index().rename(columns={'P_bloque_kw':'P_nop'})
)
_op_m = (
    op.groupby(['bloque','fecha','mes'], observed=True)
    ['P_bloque_kw'].mean().reset_index().rename(columns={'P_bloque_kw':'P_op'})
)
ind06_daily = _nop_m.merge(_op_m, on=['bloque','fecha','mes'], how='inner')
ind06_daily['f4'] = ind06_daily['P_nop'] / ind06_daily['P_op']

ind06_monthly = (
    block_power.groupby(['bloque','mes','franja'], observed=True)
    ['P_bloque_kw'].mean().unstack('franja').reset_index()
)
ind06_monthly.columns.name = None
ind06_monthly['f4'] = ind06_monthly['no_operacional'] / ind06_monthly['operacional']
print(ind06_monthly[['bloque','mes','no_operacional','operacional','f4']].to_string(index=False))

 bloque     mes  no_operacional  operacional     f4
 3.0000 2026-01          1.7099       2.9032 0.5890
 3.0000 2026-02          1.6447       5.7534 0.2859
 3.0000 2026-03          1.8029       6.2316 0.2893
 3.0000 2026-04          1.8734       6.6858 0.2802
 3.0000 2026-05          1.8581       6.7380 0.2758
 3.0000 2026-06          2.0592       8.1858 0.2516
 4.0000 2026-01         13.8025      16.4910 0.8370
 4.0000 2026-02         13.8146      28.6914 0.4815
 4.0000 2026-03         12.9989      25.9497 0.5009
 4.0000 2026-04         12.4774      25.9794 0.4803
 4.0000 2026-05         12.8740      25.5959 0.5030
 4.0000 2026-06         13.0118      32.0953 0.4054
 5.0000 2026-01         12.5273      17.0067 0.7366
 5.0000 2026-02         12.0953      34.1072 0.3546
 5.0000 2026-03         11.5695      32.1834 0.3595
 5.0000 2026-04         13.6968      38.2601 0.3580
 5.0000 2026-05         13.3559      36.9082 0.3619
 5.0000 2026-06         14.8030      45.6847 0.3240
 7.0000 2026

### IND-07 — Emisiones de CO₂ (tCO₂e)
**Fórmula:** `CO₂ = FE_2025 × Σ(E_kWh)` · FE_2025 = 0.000097018 tCO₂e/kWh (XM, 30-ene-2026)  
⚠ El factor 0.18 tCO₂/MWh de reportes anteriores está **desactualizado**.

In [12]:
ind07_monthly = (
    daily_bloque.groupby(['bloque','mes'], observed=True)
    ['E_dia_kwh'].sum().reset_index().rename(columns={'E_dia_kwh':'E_mes_kwh'})
)
ind07_monthly['CO2_tCO2e'] = ind07_monthly['E_mes_kwh'] * FE_2025

ind07_campus = (
    ind07_monthly.groupby('mes', observed=True)
    [['E_mes_kwh','CO2_tCO2e']].sum().reset_index()
)
ind07_campus['bloque'] = 'CAMPUS_TOTAL'
print('Por bloque:')
print(ind07_monthly[['bloque','mes','E_mes_kwh','CO2_tCO2e']].to_string(index=False))
print('\nCampus total:')
print(ind07_campus.to_string(index=False))

Por bloque:
 bloque     mes  E_mes_kwh  CO2_tCO2e
 3.0000 2026-01    69.5426     0.0067
 3.0000 2026-02  2990.8532     0.2902
 3.0000 2026-03  3270.3949     0.3173
 3.0000 2026-04  3469.2174     0.3366
 3.0000 2026-05  3731.4633     0.3620
 3.0000 2026-06   762.4150     0.0740
 4.0000 2026-01   436.8864     0.0424
 4.0000 2026-02 16203.6705     1.5720
 4.0000 2026-03 14788.4840     1.4347
 4.0000 2026-04 14838.4368     1.4396
 4.0000 2026-05 15618.3628     1.5153
 4.0000 2026-06  3163.1188     0.3069
 5.0000 2026-01   430.7544     0.0418
 5.0000 2026-02 18237.0588     1.7693
 5.0000 2026-03 16679.6612     1.6182
 5.0000 2026-04 20595.7310     1.9982
 5.0000 2026-05 21190.5708     2.0559
 5.0000 2026-06  4371.9188     0.4242
 7.0000 2026-01  1179.4325     0.1144
 7.0000 2026-02 33980.3687     3.2967
 7.0000 2026-03 33657.1386     3.2653
 7.0000 2026-04 33350.4315     3.2356
 7.0000 2026-05 38308.2930     3.7166
 7.0000 2026-06  6719.9274     0.6520
 8.0000 2026-01   490.6952     0.0476


### IND-08 — IGS — Yield Factor FV ⚠ PENDIENTE
**Bloqueado por:** Requiere: (1) energyproducedtoday de etfroniusinverter/etenphaseinverter; (2) P_instalada kWp Bloques 10 y 11.

In [13]:
flag_pendiente('IND-08 — IGS — Yield Factor FV', 'Requiere: (1) energyproducedtoday de etfroniusinverter/etenphaseinverter; (2) P_instalada kWp Bloques 10 y 11.')

⚠ PENDIENTE — IND-08 — IGS — Yield Factor FV:
  Requiere: (1) energyproducedtoday de etfroniusinverter/etenphaseinverter; (2) P_instalada kWp Bloques 10 y 11.


### IND-09 — TCP — Temperatura crítica de panel ⚠ PENDIENTE
**Bloqueado por:** Requiere: paneltemperature y ambienttemperature del sensor Fronius (etfroniussensorcard).

In [14]:
flag_pendiente('IND-09 — TCP — Temperatura crítica de panel', 'Requiere: paneltemperature y ambienttemperature del sensor Fronius (etfroniussensorcard).')

⚠ PENDIENTE — IND-09 — TCP — Temperatura crítica de panel:
  Requiere: paneltemperature y ambienttemperature del sensor Fronius (etfroniussensorcard).


### IND-10 — EB — Eficiencia de batería ⚠ PENDIENTE
**Bloqueado por:** Requiere: energyfrombattery y energytobattery del Inversor_Baterías.

In [15]:
flag_pendiente('IND-10 — EB — Eficiencia de batería', 'Requiere: energyfrombattery y energytobattery del Inversor_Baterías.')

⚠ PENDIENTE — IND-10 — EB — Eficiencia de batería:
  Requiere: energyfrombattery y energytobattery del Inversor_Baterías.


### IND-11 — Ahorro de energía ⚠ PENDIENTE
**Bloqueado por:** Requiere línea base ≥ 12 meses ajustada por N_usuarios y temperatura (ISO 50001 Anexo B).

In [16]:
flag_pendiente('IND-11 — Ahorro de energía', 'Requiere línea base ≥ 12 meses ajustada por N_usuarios y temperatura (ISO 50001 Anexo B).')

⚠ PENDIENTE — IND-11 — Ahorro de energía:
  Requiere línea base ≥ 12 meses ajustada por N_usuarios y temperatura (ISO 50001 Anexo B).


### IND-12 — Desbalance de tensión (VU, %)
**Fórmula:** `VU = max(|v1−v̄|, |v2−v̄|, |v3−v̄|) / v̄ × 100` · método NEMA MG-1  
**Umbrales:** objetivo < 2 % · alerta ≥ 3 % (IEEE 1159:2019)

In [17]:
df['v_mean'] = (df['v1'] + df['v2'] + df['v3']) / 3.0
df['VU_pct'] = (
    df[['v1','v2','v3']].sub(df['v_mean'], axis=0).abs().max(axis=1)
    / df['v_mean'] * 100.0
)

ind12_hourly = df[['entity_id','bloque','time_index_colombia','hora','fecha','mes','VU_pct']].copy()

ind12_monthly = (
    ind12_hourly.groupby(['bloque','mes'], observed=True)
    .agg(
        VU_media=('VU_pct','mean'),
        VU_max=('VU_pct','max'),
        horas_alerta=('VU_pct', lambda x: (x >= UMBRAL_DESBALANCE_ALERTA).sum()),
        horas_total=('VU_pct','count')
    ).reset_index()
)
ind12_monthly['pct_horas_alerta'] = (
    ind12_monthly['horas_alerta'] / ind12_monthly['horas_total'] * 100).round(1)
ind12_monthly['estado'] = ind12_monthly['VU_media'].apply(
    lambda v: estado_icon(v, UMBRAL_DESBALANCE_OBJETIVO, UMBRAL_DESBALANCE_ALERTA, False))

print(ind12_monthly[['bloque','mes','VU_media','VU_max','horas_alerta','pct_horas_alerta','estado']].to_string(index=False))

 bloque     mes  VU_media  VU_max  horas_alerta  pct_horas_alerta estado
 3.0000 2026-01    0.3490  0.3868             0            0.0000     OK
 3.0000 2026-02    0.3543  0.4599             0            0.0000     OK
 3.0000 2026-03    0.3931  0.4994             0            0.0000     OK
 3.0000 2026-04    0.3645  0.4589             0            0.0000     OK
 3.0000 2026-05    0.3463  0.8402             0            0.0000     OK
 3.0000 2026-06    0.4155  0.5982             0            0.0000     OK
 4.0000 2026-01    0.4965  0.5778             0            0.0000     OK
 4.0000 2026-02    0.4473  0.6977             0            0.0000     OK
 4.0000 2026-03    0.4631  0.6134             0            0.0000     OK
 4.0000 2026-04    0.4768  0.6339             0            0.0000     OK
 4.0000 2026-05    0.3856  0.9040             0            0.0000     OK
 4.0000 2026-06    0.4337  0.5569             0            0.0000     OK
 5.0000 2026-01    0.4013  0.4705             0    

### IND-13 — FD (Factor de Diversidad)
**Fórmula:** `FD = Σmax(P_i) / max(ΣP_i)` · campus total, mensual  
FD > 1 → picos individuales no ocurren simultáneamente.

In [18]:
# Suma de picos individuales por medidor
_meter_peaks = (
    df.groupby(['entity_id','mes'], observed=True)
    ['activepower_kw'].max().reset_index()
)
_num = _meter_peaks.groupby('mes', observed=True)['activepower_kw'].sum().rename('sum_picos_kw')

# Máximo de la potencia simultánea del campus
_campus_ts = (
    df.groupby(['time_index_colombia','mes'], observed=True)
    ['activepower_kw'].sum().reset_index()
)
_den = _campus_ts.groupby('mes', observed=True)['activepower_kw'].max().rename('max_campus_kw')

ind13_monthly = pd.concat([_num, _den], axis=1).reset_index()
ind13_monthly['FD'] = ind13_monthly['sum_picos_kw'] / ind13_monthly['max_campus_kw']
print(ind13_monthly[['mes','sum_picos_kw','max_campus_kw','FD']].to_string(index=False))

    mes  sum_picos_kw  max_campus_kw     FD
2026-01      368.0911       334.7854 1.0995
2026-02      608.4764       474.4755 1.2824
2026-03      671.5138       544.3787 1.2335
2026-04      714.8500       565.2234 1.2647
2026-05      759.9046       613.2403 1.2392
2026-06      578.8383       486.8576 1.1889


## 5. KPIs

### KPI-01 — Consumo por metro cuadrado (kWh/m²)
**Fórmula:** `Σ(E_kWh) / Área_bloque` · mensual · umbral ref.: 8–25 kWh/m²·mes (UPME PGEE)  
**Estado:** REAL

In [19]:
kpi01 = (
    daily_bloque.groupby(['bloque','mes'], observed=True)
    ['E_dia_kwh'].sum().reset_index().rename(columns={'E_dia_kwh':'E_mes_kwh'})
)
kpi01['area_m2'] = kpi01['bloque'].map(AREAS_BLOQUE)
kpi01 = kpi01.dropna(subset=['area_m2'])
kpi01['kWh_m2'] = kpi01['E_mes_kwh'] / kpi01['area_m2']
kpi01['estado'] = kpi01['kWh_m2'].apply(lambda v: 'OK' if v<=25 else ('AVISO' if v<=35 else 'ALERTA'))
print(kpi01[['bloque','mes','E_mes_kwh','area_m2','kWh_m2','estado']].to_string(index=False))

 bloque     mes  E_mes_kwh    area_m2  kWh_m2 estado
 3.0000 2026-01    69.5426  4778.6800  0.0146     OK
 3.0000 2026-02  2990.8532  4778.6800  0.6259     OK
 3.0000 2026-03  3270.3949  4778.6800  0.6844     OK
 3.0000 2026-04  3469.2174  4778.6800  0.7260     OK
 3.0000 2026-05  3731.4633  4778.6800  0.7809     OK
 3.0000 2026-06   762.4150  4778.6800  0.1595     OK
 4.0000 2026-01   436.8864 10309.8900  0.0424     OK
 4.0000 2026-02 16203.6705 10309.8900  1.5717     OK
 4.0000 2026-03 14788.4840 10309.8900  1.4344     OK
 4.0000 2026-04 14838.4368 10309.8900  1.4392     OK
 4.0000 2026-05 15618.3628 10309.8900  1.5149     OK
 4.0000 2026-06  3163.1188 10309.8900  0.3068     OK
 5.0000 2026-01   430.7544 10008.8700  0.0430     OK
 5.0000 2026-02 18237.0588 10008.8700  1.8221     OK
 5.0000 2026-03 16679.6612 10008.8700  1.6665     OK
 5.0000 2026-04 20595.7310 10008.8700  2.0577     OK
 5.0000 2026-05 21190.5708 10008.8700  2.1172     OK
 5.0000 2026-06  4371.9188 10008.8700  0.4368 

### KPI-02 — Intensidad por usuario (kWh/usuario·mes) ⚠ DEMO
**Bloqueado por:** `N_USUARIOS_ACTIVOS` no definido institucionalmente · ref = 3 500 usuarios

In [20]:
_N_REF = 3500  # DEMO_MODE: definicion formal pendiente | ref=3500 users
kpi02 = (
    daily_bloque.groupby('mes', observed=True)
    ['E_dia_kwh'].sum().reset_index().rename(columns={'E_dia_kwh':'E_campus_kwh'})
)
kpi02['N_usuarios_ref_DEMO'] = _N_REF
kpi02['kWh_usuario_DEMO'] = kpi02['E_campus_kwh'] / _N_REF
kpi02['modo'] = 'DEMO — N_usuarios sin confirmar'
flag_pendiente('KPI-02', f'N_USUARIOS_ACTIVOS no definido. Referencia provisoria = {_N_REF} usuarios.')
print(kpi02.to_string(index=False))

⚠ PENDIENTE — KPI-02:
  N_USUARIOS_ACTIVOS no definido. Referencia provisoria = 3500 usuarios.
    mes  E_campus_kwh  N_usuarios_ref_DEMO  kWh_usuario_DEMO                            modo
2026-01     5514.6485                 3500            1.5756 DEMO — N_usuarios sin confirmar
2026-02   172379.2427                 3500           49.2512 DEMO — N_usuarios sin confirmar
2026-03   247073.1078                 3500           70.5923 DEMO — N_usuarios sin confirmar
2026-04   201274.1650                 3500           57.5069 DEMO — N_usuarios sin confirmar
2026-05   213822.0762                 3500           61.0920 DEMO — N_usuarios sin confirmar
2026-06    37696.9843                 3500           10.7706 DEMO — N_usuarios sin confirmar


### KPI-03 — Pico de demanda absoluto (kW + timestamp)
**Estado:** REAL · umbral = media+1σ (definir tras primer año)

In [21]:
kpi03_rows = []
for (bloque, mes), grp in block_power.groupby(['bloque','mes'], observed=True):
    idx = grp['P_bloque_kw'].idxmax()
    r = grp.loc[idx]
    kpi03_rows.append({'bloque':bloque,'mes':mes,'D_pico_kw':r['P_bloque_kw'],
                       'fecha_pico':r['fecha'],'hora_pico':int(r['hora'])})
kpi03 = pd.DataFrame(kpi03_rows)
print(kpi03.to_string(index=False))

 bloque     mes  D_pico_kw fecha_pico  hora_pico
 3.0000 2026-01     4.4463 2026-01-31         11
 3.0000 2026-02    11.7887 2026-02-26         14
 3.0000 2026-03    15.1171 2026-03-13         14
 3.0000 2026-04    13.4860 2026-04-13         14
 3.0000 2026-05    16.2394 2026-05-27         14
 3.0000 2026-06    13.9731 2026-06-04         14
 4.0000 2026-01    19.2148 2026-01-31         13
 4.0000 2026-02    59.2032 2026-02-10         10
 4.0000 2026-03    61.0139 2026-03-02         11
 4.0000 2026-04    54.2526 2026-04-07         11
 4.0000 2026-05    63.4569 2026-05-28         12
 4.0000 2026-06    53.5272 2026-06-03         12
 5.0000 2026-01    21.1488 2026-01-30         19
 5.0000 2026-02    71.9180 2026-02-23         10
 5.0000 2026-03    72.2246 2026-03-18         10
 5.0000 2026-04    80.0197 2026-04-15         10
 5.0000 2026-05    85.8299 2026-05-11         12
 5.0000 2026-06    79.2486 2026-06-02         11
 7.0000 2026-01    50.6799 2026-01-31         12
 7.0000 2026-02    8

### KPI-04 — Ahorro energético verificado (%) ⚠ DEMO
**Bloqueado por:** sin línea base real. La fórmula se ilustra con `E_base = E_actual × 1.03`.

In [22]:
flag_pendiente('KPI-04','Requiere línea base ≥ 12 meses (ISO 50001 Anexo B).')
kpi04_demo = (
    daily_bloque.groupby(['bloque','mes'], observed=True)
    ['E_dia_kwh'].sum().reset_index().rename(columns={'E_dia_kwh':'E_actual_kwh'})
)
kpi04_demo['E_base_DEMO_kwh'] = kpi04_demo['E_actual_kwh'] * 1.03  # DEMO_MODE | ref=prior×1.03
kpi04_demo['ahorro_pct_DEMO'] = (1 - kpi04_demo['E_actual_kwh']/kpi04_demo['E_base_DEMO_kwh'])*100
kpi04_demo['modo'] = 'DEMO'
print(kpi04_demo[['bloque','mes','E_actual_kwh','ahorro_pct_DEMO','modo']].to_string(index=False))

⚠ PENDIENTE — KPI-04:
  Requiere línea base ≥ 12 meses (ISO 50001 Anexo B).
 bloque     mes  E_actual_kwh  ahorro_pct_DEMO modo
 3.0000 2026-01       69.5426           2.9126 DEMO
 3.0000 2026-02     2990.8532           2.9126 DEMO
 3.0000 2026-03     3270.3949           2.9126 DEMO
 3.0000 2026-04     3469.2174           2.9126 DEMO
 3.0000 2026-05     3731.4633           2.9126 DEMO
 3.0000 2026-06      762.4150           2.9126 DEMO
 4.0000 2026-01      436.8864           2.9126 DEMO
 4.0000 2026-02    16203.6705           2.9126 DEMO
 4.0000 2026-03    14788.4840           2.9126 DEMO
 4.0000 2026-04    14838.4368           2.9126 DEMO
 4.0000 2026-05    15618.3628           2.9126 DEMO
 4.0000 2026-06     3163.1188           2.9126 DEMO
 5.0000 2026-01      430.7544           2.9126 DEMO
 5.0000 2026-02    18237.0588           2.9126 DEMO
 5.0000 2026-03    16679.6612           2.9126 DEMO
 5.0000 2026-04    20595.7310           2.9126 DEMO
 5.0000 2026-05    21190.5708           

### KPI-05 — Emisiones CO₂ (tCO₂e)
**Estado:** REAL · meta: reducción ≥ 3 % anual (Ley 2169/2021)

In [23]:
kpi05 = ind07_monthly[['bloque','mes','E_mes_kwh','CO2_tCO2e']].copy()
kpi05['FE'] = FE_2025
kpi05['meta_pct_anual'] = 3.0
print(kpi05.to_string(index=False))
print('\nCampus total:')
print(ind07_campus.to_string(index=False))

 bloque     mes  E_mes_kwh  CO2_tCO2e     FE  meta_pct_anual
 3.0000 2026-01    69.5426     0.0067 0.0001          3.0000
 3.0000 2026-02  2990.8532     0.2902 0.0001          3.0000
 3.0000 2026-03  3270.3949     0.3173 0.0001          3.0000
 3.0000 2026-04  3469.2174     0.3366 0.0001          3.0000
 3.0000 2026-05  3731.4633     0.3620 0.0001          3.0000
 3.0000 2026-06   762.4150     0.0740 0.0001          3.0000
 4.0000 2026-01   436.8864     0.0424 0.0001          3.0000
 4.0000 2026-02 16203.6705     1.5720 0.0001          3.0000
 4.0000 2026-03 14788.4840     1.4347 0.0001          3.0000
 4.0000 2026-04 14838.4368     1.4396 0.0001          3.0000
 4.0000 2026-05 15618.3628     1.5153 0.0001          3.0000
 4.0000 2026-06  3163.1188     0.3069 0.0001          3.0000
 5.0000 2026-01   430.7544     0.0418 0.0001          3.0000
 5.0000 2026-02 18237.0588     1.7693 0.0001          3.0000
 5.0000 2026-03 16679.6612     1.6182 0.0001          3.0000
 5.0000 2026-04 20595.73

### KPI-06 — Performance Ratio FV (%) ⚠ DEMO
**Bloqueado por:** datos Fronius/Enphase · irradiancia sin resolución confirmada · kWp sin confirmar

In [24]:
flag_pendiente('KPI-06','Requiere energyproducedtoday (Fronius/Enphase) + solarirradiation (W/m²) + P_instalada kWp.')

⚠ PENDIENTE — KPI-06:
  Requiere energyproducedtoday (Fronius/Enphase) + solarirradiation (W/m²) + P_instalada kWp.


### KPI-07 — Autosuficiencia solar (%) ⚠ DEMO
**Bloqueado por:** datos inversores FV + verificar exportación etinverterxw

In [25]:
flag_pendiente('KPI-07','Requiere energyproducedtoday + verificar exportación a red en etinverterxw.')

⚠ PENDIENTE — KPI-07:
  Requiere energyproducedtoday + verificar exportación a red en etinverterxw.


### KPI-08 — Load Factor mensual
**Fórmula:** `LF = mean(P) / max(P)` · período mensual · umbral ≥ 0.65 · **Estado:** REAL

In [26]:
kpi08 = ind01_monthly[['bloque','mes','P_mean','P_max','LF']].copy()
kpi08['umbral'] = UMBRAL_LF_ORIENTADOR
kpi08['estado'] = kpi08['LF'].apply(lambda v: estado_icon(v, UMBRAL_LF_ORIENTADOR, UMBRAL_LF_ORIENTADOR))
print(kpi08[['bloque','mes','LF','estado']].to_string(index=False))

 bloque     mes     LF estado
 3.0000 2026-01 0.5604 ALERTA
 3.0000 2026-02 0.3701 ALERTA
 3.0000 2026-03 0.3138 ALERTA
 3.0000 2026-04 0.3762 ALERTA
 3.0000 2026-05 0.3145 ALERTA
 3.0000 2026-06 0.4308 ALERTA
 4.0000 2026-01 0.8100     OK
 4.0000 2026-02 0.3996 ALERTA
 4.0000 2026-03 0.3540 ALERTA
 4.0000 2026-04 0.3955 ALERTA
 4.0000 2026-05 0.3363 ALERTA
 4.0000 2026-06 0.4736 ALERTA
 5.0000 2026-01 0.7311     OK
 5.0000 2026-02 0.3707 ALERTA
 5.0000 2026-03 0.3497 ALERTA
 5.0000 2026-04 0.3730 ALERTA
 5.0000 2026-05 0.3382 ALERTA
 5.0000 2026-06 0.4387 ALERTA
 7.0000 2026-01 0.8352     OK
 7.0000 2026-02 0.6075 ALERTA
 7.0000 2026-03 0.6318 ALERTA
 7.0000 2026-04 0.5842 ALERTA
 7.0000 2026-05 0.5614 ALERTA
 7.0000 2026-06 0.6626     OK
 8.0000 2026-01 0.7461     OK
 8.0000 2026-02 0.3605 ALERTA
 8.0000 2026-03 0.3588 ALERTA
 8.0000 2026-04 0.3710 ALERTA
 8.0000 2026-05 0.3656 ALERTA
 8.0000 2026-06 0.3907 ALERTA
 9.0000 2026-01 0.6775     OK
 9.0000 2026-02 0.4111 ALERTA
 9.0000 20

### KPI-09 — Índice de consumo no operacional (ICNO, %)
**Fórmula:** `ICNO = Σ(E_no_op_kWh) / Σ(E_total_kWh) × 100` · franja no_op: 22:00–05:59  
**Umbrales:** < 20 % OK · 20–30 % AVISO · > 30 % ALERTA · **Estado:** REAL

In [27]:
kpi09 = (
    df.groupby(['bloque','mes','franja'], observed=True)
    ['E_hora_wh'].sum().unstack('franja', fill_value=0).reset_index()
)
kpi09.columns.name = None
for col in ['operacional','no_operacional']:
    if col not in kpi09.columns: kpi09[col] = 0.0
kpi09['E_total_wh'] = kpi09['operacional'] + kpi09['no_operacional']
kpi09['ICNO_pct'] = (kpi09['no_operacional'] / kpi09['E_total_wh'] * 100).round(2)
kpi09['estado'] = kpi09['ICNO_pct'].apply(lambda v: 'OK' if v<20 else ('AVISO' if v<30 else 'ALERTA'))
print(kpi09[['bloque','mes','E_total_wh','no_operacional','ICNO_pct','estado']].to_string(index=False))

 bloque     mes    E_total_wh  no_operacional  ICNO_pct estado
 3.0000 2026-01    69542.6062      16436.8800   23.6400  AVISO
 3.0000 2026-02  2990853.2400     477901.3700   15.9800     OK
 3.0000 2026-03  3270394.8636     521657.5173   15.9500     OK
 3.0000 2026-04  3469217.4284     488041.8912   14.0700     OK
 3.0000 2026-05  3731463.3274     482173.4220   12.9200     OK
 3.0000 2026-06   762415.0169     235601.8685   30.9000 ALERTA
 4.0000 2026-01   436886.4092     136998.6400   31.3600 ALERTA
 4.0000 2026-02 16203670.5267    3679539.8500   22.7100  AVISO
 4.0000 2026-03 14788483.9690    3355914.9678   22.6900  AVISO
 4.0000 2026-04 14838436.8304    2876844.5470   19.3900     OK
 4.0000 2026-05 15618362.7942    3325543.3189   21.2900  AVISO
 4.0000 2026-06  3163118.7895    1098538.5885   34.7300 ALERTA
 5.0000 2026-01   430754.3679     128730.2017   29.8800  AVISO
 5.0000 2026-02 18237058.7583    3410318.6483   18.7000     OK
 5.0000 2026-03 16679661.1962    2537421.4893   15.2100

### KPI-10 — Desbalance de tensión (%)
**Umbrales:** objetivo < 2 % · alerta ≥ 3 % · **Estado:** REAL

In [28]:
kpi10 = (
    ind12_hourly.groupby(['bloque','mes'], observed=True)
    .agg(
        VU_media=('VU_pct','mean'),
        VU_max=('VU_pct','max'),
        horas_alerta=('VU_pct', lambda x: (x >= UMBRAL_DESBALANCE_ALERTA).sum()),
        horas_total=('VU_pct','count')
    ).reset_index()
)
kpi10['pct_horas_alerta'] = (kpi10['horas_alerta']/kpi10['horas_total']*100).round(1)
kpi10['estado'] = kpi10['VU_media'].apply(
    lambda v: estado_icon(v, UMBRAL_DESBALANCE_OBJETIVO, UMBRAL_DESBALANCE_ALERTA, False))
print(kpi10[['bloque','mes','VU_media','VU_max','horas_alerta','pct_horas_alerta','estado']].to_string(index=False))

 bloque     mes  VU_media  VU_max  horas_alerta  pct_horas_alerta estado
 3.0000 2026-01    0.3490  0.3868             0            0.0000     OK
 3.0000 2026-02    0.3543  0.4599             0            0.0000     OK
 3.0000 2026-03    0.3931  0.4994             0            0.0000     OK
 3.0000 2026-04    0.3645  0.4589             0            0.0000     OK
 3.0000 2026-05    0.3463  0.8402             0            0.0000     OK
 3.0000 2026-06    0.4155  0.5982             0            0.0000     OK
 4.0000 2026-01    0.4965  0.5778             0            0.0000     OK
 4.0000 2026-02    0.4473  0.6977             0            0.0000     OK
 4.0000 2026-03    0.4631  0.6134             0            0.0000     OK
 4.0000 2026-04    0.4768  0.6339             0            0.0000     OK
 4.0000 2026-05    0.3856  0.9040             0            0.0000     OK
 4.0000 2026-06    0.4337  0.5569             0            0.0000     OK
 5.0000 2026-01    0.4013  0.4705             0    

### KPI-11 — Factor de potencia total
**Fórmula:** `totalpowerfactor` directo del medidor · mínimo del mes para alertas · **Estado:** REAL

In [29]:
_fp = (
    df.groupby(['entity_id','bloque','mes'], observed=True)
    ['totalpowerfactor'].agg(FP_min='min', FP_media='mean').reset_index()
)
kpi11 = (
    _fp.groupby(['bloque','mes'], observed=True)
    .agg(FP_min=('FP_min','min'), FP_media=('FP_media','mean')).reset_index()
)
kpi11['estado'] = kpi11['FP_min'].apply(lambda v: estado_icon(v, UMBRAL_FP_OBJETIVO, UMBRAL_FP_ALERTA))
print(kpi11[['bloque','mes','FP_min','FP_media','estado']].to_string(index=False))

 bloque     mes  FP_min  FP_media estado
 3.0000 2026-01  0.9444    0.9698     OK
 3.0000 2026-02  0.9392    0.9755     OK
 3.0000 2026-03  0.8997    0.9722  AVISO
 3.0000 2026-04  0.9503    0.9772     OK
 3.0000 2026-05  0.8991    0.9701  AVISO
 3.0000 2026-06  0.9593    0.9759     OK
 4.0000 2026-01  0.9100    0.9497     OK
 4.0000 2026-02  0.8852    0.9567  AVISO
 4.0000 2026-03  0.8771    0.9567  AVISO
 4.0000 2026-04  0.8453    0.9558 ALERTA
 4.0000 2026-05  0.8067    0.9580 ALERTA
 4.0000 2026-06  0.9041    0.9646     OK
 5.0000 2026-01  0.9740    0.9834     OK
 5.0000 2026-02  0.9436    0.9788     OK
 5.0000 2026-03  0.9326    0.9787     OK
 5.0000 2026-04  0.9418    0.9745     OK
 5.0000 2026-05  0.9300    0.9720     OK
 5.0000 2026-06  0.9478    0.9801     OK
 7.0000 2026-01  0.8184    0.9293 ALERTA
 7.0000 2026-02  0.7748    0.9349 ALERTA
 7.0000 2026-03  0.7019    0.9269 ALERTA
 7.0000 2026-04  0.7587    0.9278 ALERTA
 7.0000 2026-05  0.7591    0.9283 ALERTA
 7.0000 2026-06 

## 6. Generación del Excel

**Archivo:** `resultados_e-visor.xlsx` · Hojas: `Indicadores`, `KPIs`

In [ ]:
OUTPUT = Path('resultados_e-visor.xlsx')

# ─── Hoja Indicadores ───────────────────────────────────────────────────────
def _col(df_, ind, desc, unidad, val_col, extra_cols=None):
    t = df_.copy()
    t.insert(0,'indicador',ind)
    t['descripcion'] = desc
    t['unidad'] = unidad
    t = t.rename(columns={val_col:'valor'})
    base = ['indicador','descripcion','bloque','mes','valor','unidad']
    if extra_cols: base += extra_cols
    return t[[c for c in base if c in t.columns]]

frames_ind = []

# IND-01 a IND-06: diario
for ind_id, df_, vcol, desc in [
    ('IND-01', ind01_daily, 'LF',  'Load Factor (diario)'),
    ('IND-02', ind02_daily, 'PAR', 'Peak-to-Average Ratio (diario)'),
    ('IND-03', ind03_daily, 'f1',  'Uniformidad operacional f1 (diario)'),
    ('IND-04', ind04_daily, 'f2',  'Coef. variacion f2 (diario)'),
    ('IND-05', ind05_daily, 'f3',  'Min-to-mean f3 (diario)'),
    ('IND-06', ind06_daily, 'f4',  'Factor carga no-op f4 (diario)'),
]:
    t = df_[['bloque','fecha','mes', vcol]].copy()
    t.insert(0,'indicador',ind_id)
    t['descripcion'] = desc
    t['unidad'] = 'adimensional'
    t = t.rename(columns={vcol:'valor'})
    frames_ind.append(t[['indicador','descripcion','bloque','fecha','mes','valor','unidad']])

# IND-07 CO2
t7 = ind07_monthly[['bloque','mes','CO2_tCO2e']].copy()
t7['fecha'] = pd.NaT; t7.insert(0,'indicador','IND-07')
t7['descripcion']='Emisiones CO2 (mensual)'; t7['unidad']='tCO2e'
t7=t7.rename(columns={'CO2_tCO2e':'valor'})
frames_ind.append(t7[['indicador','descripcion','bloque','fecha','mes','valor','unidad']])

# IND-08..11 PENDIENTE
for i, d in [
    ('IND-08','IGS — Yield Factor FV — PENDIENTE: datos FV + kWp'),
    ('IND-09','TCP — Temp panel — PENDIENTE: sensor Fronius'),
    ('IND-10','EB — Eficiencia bateria — PENDIENTE: Inversor_Baterias'),
    ('IND-11','Ahorro energia — PENDIENTE: linea base >= 12 meses'),
]:
    frames_ind.append(pd.DataFrame([{'indicador':i,'descripcion':d,'bloque':'N/A',
        'fecha':pd.NaT,'mes':'N/A','valor':'PENDIENTE','unidad':'N/A'}]))

# IND-12 VU horario
t12 = ind12_hourly[['entity_id','bloque','time_index_colombia','mes','VU_pct']].copy()
t12['fecha'] = t12['time_index_colombia'].apply(lambda x: x.date())
t12.drop(columns='time_index_colombia',inplace=True)
t12.insert(0,'indicador','IND-12'); t12['descripcion']='Desbalance tension VU (horario)'; t12['unidad']='%'
t12=t12.rename(columns={'VU_pct':'valor'})
frames_ind.append(t12[['indicador','descripcion','bloque','fecha','mes','valor','unidad']])

# IND-13 FD
t13 = ind13_monthly[['mes','FD']].copy()
t13['bloque']='CAMPUS_TOTAL'; t13['fecha']=pd.NaT
t13.insert(0,'indicador','IND-13'); t13['descripcion']='Factor de Diversidad (campus)'; t13['unidad']='adimensional'
t13=t13.rename(columns={'FD':'valor'})
frames_ind.append(t13[['indicador','descripcion','bloque','fecha','mes','valor','unidad']])

df_ind = pd.concat(frames_ind, ignore_index=True)
df_ind['mes'] = df_ind['mes'].astype(str)
df_ind['fecha'] = pd.to_datetime(df_ind['fecha'], errors='coerce')

# ─── Hoja KPIs ─────────────────────────────────────────────────────────────
frames_kpi = []

def _krow(df_, kpi, desc, unidad, vcol, estado_col=None):
    t = df_.copy()
    t.insert(0,'kpi',kpi); t['descripcion']=desc; t['unidad']=unidad
    t=t.rename(columns={vcol:'valor'})
    cols=['kpi','descripcion','bloque','mes','valor','unidad']
    if estado_col and estado_col in t.columns: cols.append('estado')
        
    return t[[c for c in cols if c in t.columns]]

frames_kpi.append(_krow(kpi01[['bloque','mes','kWh_m2','estado']],'KPI-01','Consumo/m2 (mensual)','kWh/m2','kWh_m2','estado'))

t = kpi02[['mes','kWh_usuario_DEMO','modo']].copy()
t['bloque']='CAMPUS_TOTAL'
t=t.rename(columns={'kWh_usuario_DEMO':'valor','modo':'estado'})
t.insert(0,'kpi','KPI-02'); t['descripcion']='Intensidad/usuario DEMO'; t['unidad']='kWh/usuario'
frames_kpi.append(t[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

t3=kpi03[['bloque','mes','D_pico_kw','fecha_pico','hora_pico']].copy()
t3.insert(0,'kpi','KPI-03'); t3['descripcion']='Pico demanda (mensual)'; t3['unidad']='kW'; t3['estado']='Umbral TBD'
t3=t3.rename(columns={'D_pico_kw':'valor'})
frames_kpi.append(t3[['kpi','descripcion','bloque','mes','valor','unidad','estado','fecha_pico','hora_pico']])

t4=kpi04_demo[['bloque','mes','ahorro_pct_DEMO','modo']].copy()
t4.insert(0,'kpi','KPI-04'); t4['descripcion']='Ahorro energetico DEMO'; t4['unidad']='%'
t4=t4.rename(columns={'ahorro_pct_DEMO':'valor','modo':'estado'})
frames_kpi.append(t4[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

t5=kpi05[['bloque','mes','CO2_tCO2e']].copy()
t5.insert(0,'kpi','KPI-05'); t5['descripcion']='Emisiones CO2 (mensual)'; t5['unidad']='tCO2e'; t5['estado']='Meta >= 3% anual'
t5=t5.rename(columns={'CO2_tCO2e':'valor'})
frames_kpi.append(t5[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

for kid, desc in [('KPI-06','PR FV DEMO'),('KPI-07','Autosuficiencia solar DEMO')]:
    frames_kpi.append(pd.DataFrame([{'kpi':kid,'descripcion':desc,'bloque':'Bloques 10-11',
        'mes':'N/A','valor':'PENDIENTE','unidad':'%','estado':'DEMO — datos FV no disponibles'}]))

frames_kpi.append(_krow(kpi08[['bloque','mes','LF','estado']],'KPI-08','Load Factor (mensual)','adimensional','LF','estado'))

t9=kpi09[['bloque','mes','ICNO_pct','estado']].copy()
t9.insert(0,'kpi','KPI-09'); t9['descripcion']='ICNO consumo no-op (mensual)'; t9['unidad']='%'
t9=t9.rename(columns={'ICNO_pct':'valor'})
frames_kpi.append(t9[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

t10=kpi10[['bloque','mes','VU_media','estado']].copy()
t10.insert(0,'kpi','KPI-10'); t10['descripcion']='Desbalance tension media (mensual)'; t10['unidad']='%'
t10=t10.rename(columns={'VU_media':'valor'})
frames_kpi.append(t10[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

t11=kpi11[['bloque','mes','FP_min','estado']].copy()
t11.insert(0,'kpi','KPI-11'); t11['descripcion']='Factor potencia min (mensual)'; t11['unidad']='adimensional'
t11=t11.rename(columns={'FP_min':'valor'})
frames_kpi.append(t11[['kpi','descripcion','bloque','mes','valor','unidad','estado']])

df_kpi = pd.concat(frames_kpi, ignore_index=True)
df_kpi['mes'] = df_kpi['mes'].astype(str)

# ─── Escribir Excel ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    df_ind.to_excel(writer, sheet_name='Indicadores', index=False)
    df_kpi.to_excel(writer, sheet_name='KPIs', index=False)
    for sname, dfs in [('Indicadores',df_ind),('KPIs',df_kpi)]:
        ws = writer.sheets[sname]
        for col in ws.columns:
            max_len = max(len(str(col[0].value or '')),
                         max((len(str(c.value or '')) for c in col[1:]), default=0))
            ws.column_dimensions[col[0].column_letter].width = min(max_len+2, 45)

print(f'Archivo generado: {OUTPUT}')
print(f'  Indicadores: {len(df_ind)} filas')
print(f'  KPIs:        {len(df_kpi)} filas')